# Figure 3 — Distribution of reaction conditions

This notebook reproduces **Figure 3** of the manuscript: the distributions of four reaction-condition variables across the curated dataset of asymmetric organocatalytic Mannich reactions.

## Panel layout

| Panel | Quantity | Representation |
|-------|----------|----------------|
| **A** | Temperature, °C | Histogram (5 °C bins) |
| **B** | Yield, %        | Histogram (5 % bins)  |
| **C** | Solvent         | Bar chart, top-12 solvents + *mixtures* + *other* |
| **D** | Time, h         | Histogram, linear x-axis |

Solvent entries are normalized before counting: any composite solvent containing a separator (`/`, `+`, ` and `, or a ratio) is grouped into the **mixtures** category, and all xylene isomers (`xylenes`, `o-xylene`, `m-xylene`, `p-xylene`) are merged into a single **xylenes** category.

## Outputs

- `figure_03_reaction_conditions.pdf` — vector, primary submission format
- `figure_03_reaction_conditions.svg` — vector, editable
- `figure_03_reaction_conditions.png` — raster, 300 dpi

## 1. Imports

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator, AutoMinorLocator

## 2. Matplotlib style and color palette

Settings match the rest of the manuscript figures (Arial, no top/right spines, editable text in PDF/SVG). Each panel uses a distinct accent color so the four condition variables remain visually distinguishable on a printed page.

In [ ]:
plt.rcParams.update({
    'font.family':       'sans-serif',
    'font.sans-serif':   ['Arial', 'Helvetica', 'DejaVu Sans'],
    'font.size':          9,
    'axes.labelsize':    10,
    'axes.titlesize':    10,
    'axes.linewidth':     0.8,
    'axes.spines.top':    False,
    'axes.spines.right':  False,
    'axes.labelpad':      4,
    'xtick.labelsize':    8,
    'ytick.labelsize':    8,
    'xtick.major.width':  0.8,
    'ytick.major.width':  0.8,
    'xtick.minor.width':  0.6,
    'ytick.minor.width':  0.6,
    'xtick.major.size':   3.5,
    'ytick.major.size':   3.5,
    'xtick.minor.size':   2.0,
    'ytick.minor.size':   2.0,
    'xtick.direction':    'out',
    'ytick.direction':    'out',
    'legend.fontsize':    8,
    'legend.frameon':     False,
    'figure.dpi':         120,
    'savefig.dpi':        300,
    'savefig.bbox':      'tight',
    'savefig.pad_inches': 0.05,
    'pdf.fonttype':       42,
    'ps.fonttype':        42,
    'svg.fonttype':      'none',
})

COLOR_TEMP    = '#4A90C7'  # blue, temperature
COLOR_YIELD   = '#5BAA66'  # green, yield
COLOR_SOLVENT = '#9B59B6'  # purple, solvent
COLOR_TIME    = '#E58A4D'  # orange, time
COLOR_NEUTRAL = '#333333'  # dark grey for text
COLOR_GRID    = '#E0E0E0'  # light grey for gridlines

## 3. Load the dataset

In [ ]:
DATA_PATH = 'Mannich_dataset.csv'  # adjust path as needed
df = pd.read_csv(DATA_PATH)
print(f'Total reactions: {len(df)}')

## 4. Solvent normalization

Raw solvent strings are normalized before counting. The normalization preserves the chemical identity of named single solvents while collapsing two structural variants into single categories:

1. **Mixtures** — any entry containing a separator (`/`, `+`, ` and `) or a numeric ratio (`80/20`, `4:1`, etc.) is reassigned to the `mixtures` category.
2. **Xylenes** — all isomers (`xylenes`, `o-xylene`, `m-xylene`, `p-xylene`) are merged into `xylenes`.

Empty entries are dropped from the solvent panel only; the other three panels retain all reactions for which the corresponding variable is defined.

In [ ]:
MIXTURE_PATTERN = re.compile(r'[/+]|\sand\s|\d+\s*[:/]\s*\d+', flags=re.IGNORECASE)
XYLENE_PATTERN  = re.compile(r'xylene', flags=re.IGNORECASE)

def normalize_solvent(value):
    """Map a raw solvent string to a normalized category."""
    if pd.isna(value):
        return None
    s = str(value).strip()
    if not s:
        return None
    if MIXTURE_PATTERN.search(s):
        return 'mixtures'
    if XYLENE_PATTERN.search(s):
        return 'xylenes'
    return s.lower()

df['solvent_norm'] = df['solvent'].apply(normalize_solvent)
print(f'Reactions with defined solvent: {df["solvent_norm"].notna().sum()}')
print(f'Unique normalized solvents:     {df["solvent_norm"].nunique()}')

## 5. Helper functions

In [ ]:
def add_panel_label(ax, label, x=-0.18, y=1.08):
    """Add a bold panel label (e.g. 'A') in the upper-left corner."""
    ax.text(x, y, label, transform=ax.transAxes,
            fontsize=12, fontweight='bold', va='top', ha='left')


def add_stat_box(ax, lines, loc='upper right', fontsize=8):
    """Add a rounded white box with multi-line summary statistics."""
    positions = {
        'upper right': (0.97, 0.97, 'right', 'top'),
        'upper left':  (0.03, 0.97, 'left',  'top'),
    }
    x, y, ha, va = positions[loc]
    ax.text(
        x, y, '\n'.join(lines),
        transform=ax.transAxes, ha=ha, va=va,
        fontsize=fontsize, color=COLOR_NEUTRAL, linespacing=1.5,
        bbox=dict(facecolor='white', edgecolor='lightgrey',
                  linewidth=0.5, alpha=0.92,
                  pad=4, boxstyle='round,pad=0.4'),
    )


def style_axes(ax, grid_axis='y'):
    """Apply a uniform light grid placed behind the data."""
    ax.grid(axis=grid_axis, alpha=0.5, linewidth=0.5, color=COLOR_GRID)
    ax.set_axisbelow(True)

## 6. Assemble Figure 3

Four-panel grid (2 × 2). Histograms A, B, D show distributions of continuous variables with a corner box reporting *N*, mean, and median. Panel C is a horizontal bar chart of the twelve most frequent solvent categories plus *mixtures* and *other*, ordered by frequency.

In [ ]:
fig, axes = plt.subplots(
    2, 2,
    figsize=(7.4, 7.4),                # ~188 x 188 mm, double-column
    gridspec_kw={'hspace': 0.55, 'wspace': 0.32},
)
ax_A, ax_B = axes[0]
ax_C, ax_D = axes[1]

# -----------------------------------------------------------------
# Panel A - Temperature
# -----------------------------------------------------------------
temp = pd.to_numeric(df['temperature'], errors='coerce').dropna().to_numpy()

t_min = int(np.floor(temp.min() / 5) * 5)
t_max = int(np.ceil(temp.max() / 5) * 5)
ax_A.hist(temp, bins=np.arange(t_min, t_max + 5, 5),
          color=COLOR_TEMP, edgecolor='white', linewidth=0.5, alpha=0.9)
ax_A.axvline(np.mean(temp),   color=COLOR_NEUTRAL,
             linestyle='--', linewidth=1.0, alpha=0.75)
ax_A.axvline(np.median(temp), color='#C0392B',
             linestyle=':',  linewidth=1.4, alpha=0.95)

ax_A.set_xlabel('Temperature, °C')
ax_A.set_ylabel('Count')
ax_A.xaxis.set_major_locator(MultipleLocator(20))
ax_A.xaxis.set_minor_locator(MultipleLocator(5))
ax_A.yaxis.set_minor_locator(AutoMinorLocator(2))
style_axes(ax_A)
ymin, ymax = ax_A.get_ylim()
ax_A.set_ylim(ymin, ymax * 1.25)

add_stat_box(
    ax_A,
    [
        f'Mean = {np.mean(temp):.0f} °C',
        f'Median = {np.median(temp):.0f} °C',
    ],
    loc='upper right',
)
add_panel_label(ax_A, 'A')

# -----------------------------------------------------------------
# Panel B - Yield
# -----------------------------------------------------------------
yld = pd.to_numeric(df['yield'], errors='coerce').dropna().to_numpy()

ax_B.hist(yld, bins=np.arange(0, 101, 5),
          color=COLOR_YIELD, edgecolor='white', linewidth=0.5, alpha=0.9)
ax_B.axvline(np.mean(yld),   color=COLOR_NEUTRAL,
             linestyle='--', linewidth=1.0, alpha=0.75)
ax_B.axvline(np.median(yld), color='#C0392B',
             linestyle=':',  linewidth=1.4, alpha=0.95)

ax_B.set_xlabel('Yield, %')
ax_B.set_ylabel('Count')
ax_B.set_xlim(0, 100)
ax_B.xaxis.set_major_locator(MultipleLocator(20))
ax_B.xaxis.set_minor_locator(MultipleLocator(5))
ax_B.yaxis.set_minor_locator(AutoMinorLocator(2))
style_axes(ax_B)
ymin, ymax = ax_B.get_ylim()
ax_B.set_ylim(ymin, ymax * 1.25)

add_stat_box(
    ax_B,
    [
        f'Mean = {np.mean(yld):.1f} %',
        f'Median = {np.median(yld):.1f} %',
    ],
    loc='upper left',
)
add_panel_label(ax_B, 'B')

# -----------------------------------------------------------------
# Panel C - Solvent (top-12 + mixtures + other)
# -----------------------------------------------------------------
# Mapping from normalized solvent keys to display labels.
# Numerical subscripts in chemical formulas use matplotlib mathtext
# (e.g. r'CH$_2$Cl$_2$') so they render as proper subscripts in PDF/SVG.
SOLVENT_LABELS = {
    'ch2cl2':   r'CH$_2$Cl$_2$',
    'toluene':  'toluene',
    'dmso':     'DMSO',
    'thf':      'THF',
    'xylenes':  'xylenes',
    'chcl3':    r'CHCl$_3$',
    'et2o':     r'Et$_2$O',
    'ch3cn':    r'CH$_3$CN',
    'mixtures': 'mixtures',
    'dmf':      'DMF',
    'mtbe':     'MTBE',
    'dioxane':  'dioxane',
    'other':    'other',
}
def solvent_label(key):
    return SOLVENT_LABELS.get(key, key)

TOP_N = 12
solv_counts = df['solvent_norm'].dropna().value_counts()
top = solv_counts.head(TOP_N)
remainder = solv_counts.iloc[TOP_N:].sum()
if remainder > 0:
    top = pd.concat([top, pd.Series({'other': remainder})])

# Order: descending size, but keep 'other' last if present
labels = [c for c in top.index if c != 'other']
labels = sorted(labels, key=lambda c: top[c], reverse=True)
if 'other' in top.index:
    labels = labels + ['other']
values = [top[c] for c in labels]

y_pos = np.arange(len(labels))[::-1]  # largest at top
ax_C.barh(y_pos, values, color=COLOR_SOLVENT,
          edgecolor='white', linewidth=0.6, height=0.72, alpha=0.9)

total_solv = int(solv_counts.sum())
for y, val in zip(y_pos, values):
    pct = val / total_solv * 100
    ax_C.text(val + max(values) * 0.012, y, f'{val:,} ({pct:.1f}%)',
              va='center', ha='left', fontsize=7.5, color=COLOR_NEUTRAL)

ax_C.set_yticks(y_pos)
ax_C.set_yticklabels([solvent_label(c) for c in labels], fontsize=8)
ax_C.set_xlabel('Count')
ax_C.set_xlim(0, max(values) * 1.22)
ax_C.xaxis.set_minor_locator(AutoMinorLocator(2))
style_axes(ax_C, grid_axis='x')
add_panel_label(ax_C, 'C')

# -----------------------------------------------------------------
# Panel D - Time
# -----------------------------------------------------------------
time_h = pd.to_numeric(df['time'], errors='coerce').dropna().to_numpy()

# Display range covers the full dataset; bin width chosen to keep the
# histogram readable across the 0-360 h interval.
t_display = 360.0
bin_width = 8.0
bins = np.arange(0, t_display + bin_width, bin_width)

ax_D.hist(time_h, bins=bins,
          color=COLOR_TIME, edgecolor='white', linewidth=0.5, alpha=0.9)
ax_D.axvline(np.mean(time_h),   color=COLOR_NEUTRAL,
             linestyle='--', linewidth=1.0, alpha=0.75)
ax_D.axvline(np.median(time_h), color='#C0392B',
             linestyle=':',  linewidth=1.4, alpha=0.95)

ax_D.set_xlabel('Time, h')
ax_D.set_ylabel('Count')
ax_D.set_xlim(0, t_display)
ax_D.xaxis.set_major_locator(MultipleLocator(48))
ax_D.xaxis.set_minor_locator(MultipleLocator(8))
ax_D.yaxis.set_minor_locator(AutoMinorLocator(2))
style_axes(ax_D)
ymin, ymax = ax_D.get_ylim()
ax_D.set_ylim(ymin, ymax * 1.25)

add_stat_box(
    ax_D,
    [
        f'Mean = {np.mean(time_h):.1f} h',
        f'Median = {np.median(time_h):.1f} h',
    ],
    loc='upper right',
)
add_panel_label(ax_D, 'D')

plt.show()

## 7. Export

In [ ]:
for ext in ('pdf', 'svg', 'png'):
    fig.savefig(f'figure_03_reaction_conditions.{ext}',
                dpi=300 if ext == 'png' else None)